# Chorus on Colab

Runs the echo-chamber simulation against a self-hosted open-weight model.
Cost is GPU time only; the model weights are free.

**Set the runtime to a GPU first:** Runtime > Change runtime type > T4 GPU.

Two things about notebooks that matter here:

- `asyncio.run()` cannot be called from inside a running event loop, and a
  notebook always has one. So the run is launched as a **subprocess**
  (`!python run_simulation.py`), not imported into a cell.
- The Colab filesystem is **ephemeral**. Cell 2 mounts Drive so a run that
  outlives a session is not lost. Skip it only for a throwaway smoke test.


## 1. Check the GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

# vLLM needs compute capability >= 7.0.
#   T4  = 7.5  OK (no bfloat16, so --dtype half is required)
#   L4  = 8.9  OK
#   A100= 8.0  OK
#   P100= 6.0  NOT SUPPORTED, vLLM will refuse to start


## 2. Persist output to Drive (recommended)

Skip this only if you do not mind losing the run when the session ends.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUNS_DIR = '/content/drive/MyDrive/chorus_runs'
!mkdir -p "$RUNS_DIR"
print('output ->', RUNS_DIR)


## 3. Install and clone

vLLM is a large install; this takes a few minutes.


In [ ]:
!pip install -q vllm
!git clone -q https://github.com/ResearchDrafts/Agentic-LLM-Network.git
%cd Agentic-LLM-Network
!pip install -q -e .
print('ready')


## 4. Pick a model that fits the GPU

Weights at FP16, before KV cache and activations:

| Model | Weights | Fits 16 GB T4? |
|---|---|---|
| `Qwen2.5-3B-Instruct` | ~6 GB | yes, comfortably |
| `Qwen2.5-7B-Instruct` | ~15 GB | only with `--max-model-len` trimmed |
| `Qwen2.5-VL-7B-Instruct` | ~18 GB | no, needs L4/A100 or 2 GPUs |

`--max-model-len 4096` matters: prompts here run 400-800 tokens, so the
default 131K context reserves KV cache you will never use.


In [ ]:
import torch

gb = torch.cuda.get_device_properties(0).total_memory / 1e9
MODEL = 'Qwen/Qwen2.5-7B-Instruct' if gb > 20 else 'Qwen/Qwen2.5-3B-Instruct'
print(f'{gb:.0f} GB detected -> serving {MODEL}')


## 5. Start the vLLM server

Runs in the background. The wait loop below blocks until it is actually
serving, which takes a couple of minutes on first run while weights download.


In [ ]:
import subprocess, time, urllib.request, json

server = subprocess.Popen(
    ['vllm', 'serve', MODEL, '--dtype', 'half',
     '--max-model-len', '4096', '--port', '8000'],
    stdout=open('/content/vllm.log','w'), stderr=subprocess.STDOUT)

for attempt in range(120):  # up to ~10 minutes
    try:
        with urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2) as r:
            print('server up:', json.loads(r.read())['data'][0]['id'])
            break
    except Exception:
        if server.poll() is not None:
            print('server died, last 40 lines:')
            !tail -40 /content/vllm.log
            break
        time.sleep(5)
else:
    print('timed out; check /content/vllm.log')


## 6. Write a config

`model_backend_id` must match the served name exactly, prefixed with
`hosted_vllm/`. `api_base` is required: without it litellm routes on the
prefix alone and never reaches localhost.

Start small. M=5, K=1 is five calls and proves the whole path.


In [ ]:
import yaml, pathlib

cfg = {
    'run_id': 'colab_smoke',
    'rq_target': 'RQ1_RQ2',
    'topic': 'whether remote work should be the default for office jobs',
    'alpha': 0.5,
    'M': 5, 'N': 2, 'K': 1,
    'trial_number': 1,
    'language_condition': 'english',
    'model_backend_id': f'hosted_vllm/{MODEL}',
    'api_base': 'http://localhost:8000/v1',
    'stance_scale': [1, 2, 3, 4, 5, 6, 7],
    'stance_low_label': 'remote work should never be the default',
    'stance_high_label': 'remote work should always be the default',
    'persona_pool_id': 'test_pool',
    'meme_injection': {'enabled': False},
    'seed': 42,
    'temperature': 0.7,
    'rate_limits': {},          # empty: let vLLM batch all agents
    'checkpoint_every_n_turns': 1,
}
pathlib.Path('configs/colab_smoke.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
!python run_simulation.py configs/colab_smoke.yaml --dry-run


## 7. Run it

Subprocess, not an import: a notebook already has an event loop, so
`asyncio.run()` inside a cell would raise.


In [ ]:
RUNS = RUNS_DIR if 'RUNS_DIR' in dir() else 'runs'
!python run_simulation.py configs/colab_smoke.yaml --runs-dir "$RUNS"


## 8. Look at what came out

Three things decide whether this is worth scaling up.


In [ ]:
import pandas as pd, json, os

df = pd.read_json(f'{RUNS}/colab_smoke/interactions.jsonl', lines=True)
print(df[['speaker_agent_id','turn','stance_before','stance_after','api_call_status']])
print()
print('1. all calls succeeded :', (df.api_call_status == 'success').all())
print('2. stances moved       :', (df.stance_before != df.stance_after).any())
print('3. mean prompt tokens  :', int(df.prompt_token_count.mean()))
print()
print('--- one agent\'s reasoning ---')
print(df.reason_text.iloc[0][:400])


### Reading that

- **`failed_logged_null` rows** mean the model is not emitting the
  `STANCE: <n>` line. Some of this is normal from a 3B model. A lot of it
  means the model is too weak for this task, which matters most for
  Hinglish, where formatting is hardest.
- **Stances never moving** points at alpha or the prompt, not the model.
- **`reason_text`** should sound like the persona and address the topic.

## 9. Scaling up

Raise `M`, `K`, and `trial_number`, then sweep `language_condition` across
`english` and `hinglish` with the **same seed**, so neighbour sampling is
held constant and only language varies.

Before a real campaign, check Hinglish quality on ~100 calls. If the model
cannot produce consistent Hinglish, RQ1 measures model deficiency rather
than a language effect, and no analysis fixes that afterwards.

A run interrupted by a session timeout resumes byte-identically from its
checkpoint: re-run the same command, provided `--runs-dir` pointed at Drive.


## Shutting down


In [ ]:
server.terminate()
print('server stopped')
